# DATA110 Capstone — Predictive Maintenance for Aircraft Engines

**Student:** Aman Naresh Undirwade  
**Course:** DATA110 — Introduction to Python using Machine Learning  
**Dataset:** NASA C-MAPSS FD004  
**Task:** Early aircraft-engine failure detection using classical machine learning

This notebook is the executable implementation accompanying the capstone report and presentation.

### Research question
How does the prediction horizon influence the effectiveness of classical machine-learning models for early aircraft-engine failure detection using NASA C-MAPSS FD004?

### Workflow
1. Load and inspect FD004
2. Construct remaining useful life (RUL)
3. Create 10/20/30/50-cycle warning targets
4. Perform engine-level validation to reduce trajectory leakage
5. Compare five DATA110 algorithms
6. Select a probability threshold using an illustrative 5:1 missed-failure cost
7. Retrain Random Forest and evaluate the official test engines
8. Inspect errors and explain the final model with SHAP

> **Data note:** NASA C-MAPSS raw files are intentionally not stored in this GitHub repository. Place `train_FD004.txt`, `test_FD004.txt`, and `RUL_FD004.txt` in `data/raw/` before running the notebook.


In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
HORIZONS = (10, 20, 30, 50)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from predictive_maintenance import (
    load_fd004, add_training_rul, add_test_rul, add_warning_label, build_feature_matrix,
)
from run_experiment import make_models, engine_holdout, metrics, select_cost_threshold

print("Project root:", PROJECT_ROOT)
print("Imports completed successfully.")


In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "raw"
required_files = ["train_FD004.txt", "test_FD004.txt", "RUL_FD004.txt"]
missing = [f for f in required_files if not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        "Missing FD004 files: " + ", ".join(missing) +
        "\nPlace the NASA C-MAPSS FD004 files in: " + str(DATA_DIR)
    )
train_raw, test_raw, official_rul = load_fd004(DATA_DIR)
print("Training shape:", train_raw.shape)
print("Official test shape:", test_raw.shape)
print("Official RUL entries:", len(official_rul))
display(train_raw.head())


## 1. Data understanding

Each row is one operating cycle for an engine. The `unit` identifier links observations belonging to the same engine, so rows from one engine must not be treated as independent observations during validation.

FD004 contains 249 training engines and 248 official test engines, with 3 operating settings and 21 sensor measurements.


In [ ]:
print("Training engines:", train_raw["unit"].nunique())
print("Training observations:", len(train_raw))
print("Test engines:", test_raw["unit"].nunique())
print("Test observations:", len(test_raw))
print("Operating settings:", 3)
print("Sensor measurements:", 21)
display(train_raw.groupby("unit")["cycle"].max().describe().to_frame("max_cycle"))


In [ ]:
train = add_training_rul(train_raw)
test = add_test_rul(test_raw, official_rul)
print("Example training RUL calculation:")
display(train[["unit", "cycle", "rul"]].head(10))
print("RUL range:", int(train["rul"].min()), "to", int(train["rul"].max()))


## 2. Exploratory analysis

The first plots show how trajectories and sensor measurements evolve over operating cycles.


In [ ]:
example_unit = int(train["unit"].iloc[0])
example = train[train["unit"] == example_unit]
plt.figure(figsize=(10, 4))
plt.plot(example["cycle"], example["s13"], label="s13")
plt.plot(example["cycle"], example["s11"], label="s11")
plt.xlabel("Operating cycle")
plt.ylabel("Sensor value")
plt.title(f"Example engine trajectory — Engine {example_unit}")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(train["rul"], bins=40)
plt.xlabel("Remaining useful life (cycles)")
plt.ylabel("Observations")
plt.title("Training RUL distribution")
plt.tight_layout()
plt.show()


## 3. Warning-horizon formulation

For each selected horizon, an observation is labelled positive when RUL <= horizon. The project compares 10, 20, 30 and 50 cycles before failure.


In [ ]:
horizon_summary = []
for h in HORIZONS:
    labelled = add_warning_label(train, h)
    horizon_summary.append({"Horizon": h, "Positive observations": int(labelled["target"].sum()), "Positive rate": float(labelled["target"].mean())})
display(pd.DataFrame(horizon_summary))


## 4. Engine-level validation

A random row split can place observations from the same engine in both training and validation. The project therefore holds out complete engines using GroupShuffleSplit.


In [ ]:
fit_df, val_df = engine_holdout(train)
print("Training engines used for validation experiment:", fit_df["unit"].nunique())
print("Validation engines:", val_df["unit"].nunique())
print("Overlap in engine IDs:", len(set(fit_df["unit"]) & set(val_df["unit"])))


## 5. Model comparison

Five classical DATA110 algorithms are compared: Logistic Regression, Gaussian Naive Bayes, KNN, Decision Tree and Random Forest.


In [ ]:
def prepare(df, horizon):
    labelled = add_warning_label(df, horizon)
    X, feature_cols = build_feature_matrix(labelled)
    y = labelled["target"].to_numpy()
    return X, y, feature_cols

comparison_rows = []
for horizon in HORIZONS:
    X_train, y_train, _ = prepare(fit_df, horizon)
    X_val, y_val, _ = prepare(val_df, horizon)
    for name, model in make_models().items():
        model.fit(X_train, y_train)
        probability = model.predict_proba(X_val)[:, 1]
        result = metrics(y_val, probability, threshold=0.50)
        result["Model"] = name
        result["Horizon"] = horizon
        comparison_rows.append(result)
comparison = pd.DataFrame(comparison_rows)[["Horizon","Model","ROC_AUC","PR_AUC","Precision","Recall","F1"]].sort_values(["Horizon","PR_AUC"], ascending=[True,False])
display(comparison.round(4))


In [ ]:
pivot = comparison.pivot(index="Horizon", columns="Model", values="PR_AUC")
ax = pivot.plot(kind="bar", figsize=(11, 5))
ax.set_ylabel("PR-AUC")
ax.set_title("PR-AUC across warning horizons")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
best_row = comparison.loc[comparison["PR_AUC"].idxmax()]
print("Best validation PR-AUC:")
display(best_row.to_frame().T)


## 6. Selected configuration

The written capstone selects the 50-cycle warning horizon and Random Forest as the final classical model. Threshold selection is performed separately from the official test set using an illustrative 5:1 missed-failure cost.


In [ ]:
SELECTED_HORIZON = 50
fit_df_threshold, val_df_threshold = engine_holdout(train)
X_fit, y_fit, feature_cols = prepare(fit_df_threshold, SELECTED_HORIZON)
X_val, y_val, _ = prepare(val_df_threshold, SELECTED_HORIZON)
rf = make_models()["Random Forest"]
rf.fit(X_fit, y_fit)
val_probability = rf.predict_proba(X_val)[:, 1]
threshold_result = select_cost_threshold(y_val, val_probability, fn_cost=5.0, fp_cost=1.0)
print("Selected horizon:", SELECTED_HORIZON)
print("Selected probability threshold:", threshold_result["threshold"])
print("Validation false negatives:", threshold_result["false_negatives"])
print("Validation false positives:", threshold_result["false_positives"])


In [ ]:
rows = []
for t in [0.50, threshold_result["threshold"]]:
    rows.append({"Threshold": t, **metrics(y_val, val_probability, threshold=t)})
display(pd.DataFrame(rows).round(4))


## 7. Official FD004 test evaluation

Only after the warning horizon, model family and threshold are selected do we evaluate the official test engines.


In [ ]:
X_full, y_full, feature_cols = prepare(train, SELECTED_HORIZON)
final_rf = make_models()["Random Forest"]
final_rf.fit(X_full, y_full)
test_labelled = add_warning_label(test, SELECTED_HORIZON)
X_test_all, _, _ = prepare(test_labelled, SELECTED_HORIZON)
last_mask = test_labelled["cycle"].eq(test_labelled.groupby("unit")["cycle"].transform("max"))
X_test_last = X_test_all.loc[last_mask].reset_index(drop=True)
test_last = test_labelled.loc[last_mask].sort_values("unit").reset_index(drop=True)
test_probability = final_rf.predict_proba(X_test_last)[:, 1]
test_y = (test_last["rul"].to_numpy() <= SELECTED_HORIZON).astype(int)
chosen_threshold = float(threshold_result["threshold"])
official_metrics = metrics(test_y, test_probability, threshold=chosen_threshold)
display(pd.DataFrame([official_metrics]).round(4))


In [ ]:
official_predictions = pd.DataFrame({
    "unit": test_last["unit"].to_numpy(),
    "cycle": test_last["cycle"].to_numpy(),
    "rul": test_last["rul"].to_numpy(),
    "probability": test_probability,
    "prediction": (test_probability >= chosen_threshold).astype(int),
    "actual": test_y,
})
official_predictions["error_type"] = np.select([
    (official_predictions["actual"] == 1) & (official_predictions["prediction"] == 0),
    (official_predictions["actual"] == 0) & (official_predictions["prediction"] == 1),
], ["False Negative", "False Positive"], default="Correct")
print("Official test engines:", len(official_predictions))
print("Actual positive engines:", int(official_predictions["actual"].sum()))
print("False negatives:", int((official_predictions["error_type"] == "False Negative").sum()))
print("False positives:", int((official_predictions["error_type"] == "False Positive").sum()))
display(official_predictions.head())


In [ ]:
cm = pd.crosstab(official_predictions["actual"], official_predictions["prediction"], rownames=["Actual"], colnames=["Predicted"], dropna=False)
display(cm)


## 8. Error analysis

The threshold intentionally prioritizes recall because a missed impending failure is treated as more costly than a false alarm. The 5:1 ratio is illustrative, not an operational aircraft-maintenance cost model.


In [ ]:
error_counts = official_predictions["error_type"].value_counts()
display(error_counts.to_frame("Count"))
plt.figure(figsize=(8, 4))
error_counts.plot(kind="bar")
plt.ylabel("Number of engines")
plt.title("Official-test prediction outcomes")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 9. SHAP explainability

SHAP is used to understand which input measurements most influence the final Random Forest predictions. SHAP explains model behavior; it does not prove that an influential sensor is the physical cause of degradation.


In [ ]:
import shap
sample_n = min(500, len(X_test_last))
X_shap = X_test_last.iloc[:sample_n].copy()
rf_estimator = final_rf.named_steps["model"]
imputer = final_rf.named_steps["imputer"]
X_shap_imputed = pd.DataFrame(imputer.transform(X_shap), columns=feature_cols)
explainer = shap.TreeExplainer(rf_estimator)
shap_values = explainer.shap_values(X_shap_imputed)
if isinstance(shap_values, list):
    shap_for_positive = shap_values[1]
else:
    shap_array = np.asarray(shap_values)
    shap_for_positive = shap_array[:, :, 1] if shap_array.ndim == 3 else shap_array
mean_abs_shap = pd.Series(np.abs(shap_for_positive).mean(axis=0), index=feature_cols).sort_values(ascending=False)
display(mean_abs_shap.head(15).to_frame("mean_abs_SHAP"))


In [ ]:
top_shap = mean_abs_shap.head(15).sort_values()
plt.figure(figsize=(9, 6))
top_shap.plot(kind="barh")
plt.xlabel("Mean absolute SHAP value")
plt.title("Top model-influential features")
plt.tight_layout()
plt.show()


## 10. Reproducible outputs

Generated outputs can be stored under `results/notebook_run/`. The raw NASA benchmark is not committed to GitHub.


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "results" / "notebook_run"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(OUTPUT_DIR / "course_algorithm_engine_level_comparison_corrected.csv", index=False)
official_predictions.to_csv(OUTPUT_DIR / "official_test_engine_predictions_corrected.csv", index=False)
pd.DataFrame([{"selected_horizon": SELECTED_HORIZON, "selected_threshold": chosen_threshold, **official_metrics}]).to_csv(OUTPUT_DIR / "OFFICIAL_TEST_ENGINE_LEVEL_RESULTS_CORRECTED.csv", index=False)
print("Saved generated results to:", OUTPUT_DIR)


## Conclusion

This implementation operationalizes the capstone's main finding: warning-horizon definition matters. The study compares four horizons using leakage-aware engine-level validation, selects Random Forest as the strongest classical benchmark at the 50-cycle horizon, chooses a cost-sensitive threshold separately from the official test set, and uses SHAP to interpret model behavior.

The benchmark is simulated and should be treated as a reproducible academic study rather than evidence of certified aircraft maintenance capability.
